In [ ]:
#importing the liberary's
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import h5py
import random
import itertools
import math
import os
import pandas as pd

np.random.seed(12)

#Creating a random matrix to test the algorithms on
def random_matrix(total, fraction, length, width):
    amount= round(total*fraction)
    matrix= np.zeros((length, width), dtype=int)
    index= np.random.choice(total,amount,replace=False)
    matrix[np.unravel_index(index, (length,width))]=1
    matrix= np.expand_dims(matrix, axis=2)
    return matrix


#The functie to plot the matrices
def segmentatie_plot(data):
        #data_new = np.where(data != 1, (0,0,255), data)
        data_new= np.where(data==1,(255,0,138),data)

        img= data_new/255

        plt.imshow(img)
        plt.axis("off")
        #If you want to save the segmentationplots remove the # en give a name
        #plt.savefig(f'{output_path}\{well}_{day}.png', bbox_inches= 'tight', pad_inches=0)
        plt.show()

#Calculates the fraction of cells in the total segmentation        
def fraction(data):
        data_new = np.where(data==2,0,data)
        rows, columns, depth= data.shape
        total= rows*columns
        fractie= data_new.sum()/total
        return rows, columns, total,fractie

#Calculates and fills the rows into segments that can be divided by 16 without rest
def pad_image(image):
    height, width = image.shape
    
    # Calculates how many pixels you need to add
    remainder = width % 16
    if remainder == 0:
        return image  # al deelbaar door 16
    
    padding = 16 - remainder
    
    #Adds zeros on the right side to make it dividable by 16
    padded = np.pad(image, ((0, 0), (0, padding)), mode='constant', constant_values=0)
    
    print(f"Originele breedte: {width}")
    print(f"Padding toegevoegd: {padding}")
    print(f"Nieuwe breedte: {padded.shape[1]}")
    
    return padded

#Makes the class object
class runNode:
    def __init__(self, start, end, row):
        self.start= start
        self.end= end
        self. row= row
        self.head= self
        self.next= None
        self.tot_area= (end-start)+1
        self.size= 1
        self.label= None
        self.row_min= row
        self.row_max= row
        self.col_min= start
        self.col_max= end

#This function merges two components
def merge(curr, upper):
    a = curr.head
    b = upper.head

    if a is b:
        return b

    if a.size > b.size:
        a, b = b, a

    #Walks throug a, updates heads and rembers the last node
    node = a
    while node.next is not None:
        node.head = b
        node = node.next
    
    #Updates the last node from a
    node.head = b


    #connects the two components
    node.next = b.next   # ← last elment of a points to the chain of b
    b.next = a           # ← b start with the chain of a

    b.size += a.size
    b.tot_area += a.tot_area

    # update bounding box of b
    b.row_min = min(b.row_min, a.row_min)
    b.row_max = max(b.row_max, a.row_max)
    b.col_min = min(b.col_min, a.col_min)
    b.col_max = max(b.col_max, a.col_max)

    return b


#merge horizontal 
def merge_horizontal(all_runs, row_start, row_end, total_rows, amount=1):
    for row in range(total_rows):
        if row_start[row] is None:
            continue
        
        runs = all_runs[row_start[row]:row_end[row] + 1]
        
        for i in range(len(runs) - 1):
            curr  = runs[i]
            next_ = runs[i + 1]
            
            
            gap = amount - 1  
            if curr.end + 1 + gap >= next_.start:
                merge(curr, next_)

#Cache aanmaken
idk=16
combinations=list(itertools.product([0,1], repeat=idk))
print(combinations)

Cache={}
for wordValue in combinations:
    segments=[]
    for i in range(idk):
        if wordValue[i]==1:
            start=i
            end= i
            segments.append((start,end))
        i= len(segments)-2
        if i>= 0:
            if segments[i][1]+1>= segments[i+1][0]:
                segments[i]=(segments[i][0],segments[i+1][1])
                segments.pop(i+1)
                i+=1
            else:
                i=i-1
        
    
    Cache[wordValue]=segments

#This function find all runs
def finding_all_runs(matrix2):
    RowSegments=[]
    for row in range(len(matrix2)):
        #prev_run= None
        imageRow= matrix2[row]
        imagewidthwords= math.ceil(len(matrix2[row])/idk)
        for j in range(imagewidthwords):
            word= imageRow[(idk*j):((16*(j+1)))]
            right_format= tuple(word.flatten().tolist())
            Segments= Cache[right_format].copy()
            for segment in range(len(Segments)):
                start= Segments[segment][0]
                eind= Segments[segment][1]
                Segments[segment]= (((idk*(j))+start),((idk*(j))+eind))
                RowSegments.append(runNode(Segments[segment][0],Segments[segment][1], row))
    return RowSegments

#this is the function that find the connected components
def connected_components_labeling(RowSegments, length,connectivity, amount_rows):
 
    rowStart= [None]*length
    rowEnd= [None]*length
    
    for element in range(len(RowSegments)):
        if rowStart[RowSegments[element].row]==None:
            rowStart[RowSegments[element].row]= element
        rowEnd[RowSegments[element].row]= element
  
    merge_horizontal(RowSegments, rowStart, rowEnd, (length),amount_rows)
    if connectivity==0:
        for row in range(1,length):
            if rowStart[row]==None:
                print('niet beginnen',row)
                continue
            for amount in range((amount_rows-1),0,-1):
                previous_row= row-amount
                if previous_row<0:
                    continue
            
                if rowStart[previous_row]==None:
                    
                    continue
              
                for dif_i in RowSegments[rowStart[row]: (rowEnd[row]+1)]:
                    for dif_j in RowSegments[rowStart[previous_row]:(rowEnd[previous_row]+1)]:
                        if abs(dif_i.row-dif_j.row)>=(amount_rows-1):
                            print(f'FOUT: vergelijkt rijen die ver uit elkaar liggen!')
                            print(f' dif_i : row= {dif_i.row} start={dif_i.start} end={dif_i.end}')
                            print(f' dif_i : row= {dif_j.row} start={dif_j.start} end={dif_j.end}')
                            print(f' rijverschil:{abs((dif_i.row- dif_j.row))}')
                        if dif_i.start<= dif_j.end and dif_j.start<= dif_i.end:
                            merge(dif_i, dif_j)
    else:
        for row in range(1,length):
            if rowStart[row]==None:
            
                continue
            for amount in range((amount_rows-1),0,-1):
                previous_row= row-amount
                if previous_row<0:
                    continue
               
                if rowStart[previous_row]==None:
                    print('vergelijken', row-amount)
                    continue
              
                for dif_i in RowSegments[rowStart[row]: (rowEnd[row]+1)]:
                    for dif_j in RowSegments[rowStart[previous_row]:(rowEnd[previous_row]+1)]:
                        if abs(dif_i.row-dif_j.row)>(amount_rows-1):
                            print(f'FOUT: vergelijkt rijen die ver uit elkaar liggen!')
                            print(f' dif_i : row= {dif_i.row} start={dif_i.start} end={dif_i.end}')
                            print(f' dif_i : row= {dif_j.row} start={dif_j.start} end={dif_j.end}')
                            print(f' rijverschil:{abs((dif_i.row- dif_j.row))}')
                        row_dist = abs(dif_i.row - dif_j.row)
                        if (abs(dif_i.start - dif_j.end)==row_dist) or (abs(dif_i.start- dif_j.start)==row_dist) or (abs(dif_i.end- dif_j.start)==row_dist) or (abs(dif_i.end-dif_j.end)==row_dist) or(dif_i.start <= dif_j.end and dif_j.start<= dif_i.end):
                            merge(dif_i, dif_j)

#this function gives each component a label
def assign_labels(runs):
    current_label = 1
    seen = {}

    for run in runs:
        root = run.head
        if root not in seen:
            seen[root] = current_label
            current_label += 1
        run.label = seen[root]

    return runs

#this function plot all components
def plot_connected_components(RowsSegments, height, width):
    
    # verzamel alle unieke labels
    labels = set(node.label for node in RowsSegments if node.label is not None)
    
    # maak een willekeurige kleur per label
    random.seed(42)  # seed voor reproduceerbare kleuren
    color_map = {
        label: (random.random(), random.random(), random.random())
        for label in labels
    }
    
    # bouw een RGB afbeelding op
    image = np.ones((height, width, 3))  # wit canvas
    
    for node in RowsSegments:
        if node.label is not None:
            color = color_map[node.label]
            image[node.row, node.start:node.end + 1] = color
    
    # plot
    fig, ax = plt.subplots(figsize=(20, 20))
    ax.imshow(image)
    ax.set_title("Connected Components")
    ax.axis("off")
    
    # legenda
    patches = [
        mpatches.Patch(color=color_map[label], label=f"Component {label}")
        for label in sorted(labels)
    ]
    #ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    #plt.savefig(r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni_fotos\Segmenteren_gehele_well\A1\connected_day8_2pixels_legend", bbox_inches='tight', pad_inches=0)
    plt.savefig(f'{output_path}\{well}_{day}_.png', bbox_inches= 'tight', pad_inches=0)
    plt.show()

#this function plots the connected components with a legend
# def plot_connected_components(all_runs, height, width, min_size=1):
    
#     labels = set(
#         node.label for node in all_runs 
#         if node.head.size >= min_size
#     )
    
#     random.seed(42)
#     color_map = {
#         label: (random.random(), random.random(), random.random())
#         for label in labels
#     }
    
#     image = np.ones((height, width, 3))
    
#     for node in all_runs:
#         if node.label is not None and node.head.size >= min_size:
#             color = color_map[node.label]
#             image[node.row, node.start:node.end + 1] = color
    
#     fig, ax = plt.subplots(figsize=(40, 40))
#     ax.imshow(image)
#     ax.set_title("Connected Components")
    
#     # raster toevoegen
#     ax.set_xticks(np.arange(-0.5, width, 1), minor=True)
#     ax.set_yticks(np.arange(-0.5, height, 1), minor=True)
#     ax.grid(which='minor', color='gray', linewidth=0.5, alpha=0.5)
    
#     # labels op de assen
#     ax.set_xticks(np.arange(0, width, 5))    # elke 5 pixels een label
#     ax.set_yticks(np.arange(0, height, 5))   # elke 5 pixels een label
    
#     plt.tight_layout()
#     plt.savefig(r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni_fotos\Segmenteren_gehele_well\A2\cluster_plot_day2_2_pixel")
#     plt.show()

#this functions sets a connected component lists back into a binary matrix
def create_binary_matrix(all_runs, height, width, min_pixels=50):
    
    # maak lege matrix
    matrix = np.zeros((height, width), dtype=np.uint8)
    
    # verzamel labels die groot genoeg zijn
    valid_labels = set(
        node.label for node in all_runs 
        if node.head is node and node.tot_area >= min_pixels
    )
    
    # vul matrix in
    for node in all_runs:
        if node.label in valid_labels:
            matrix[node.row, node.start:node.end + 1] = 1
    m= np.expand_dims(matrix, axis=-1)
    
    return m

#this function merge two matrices
def merging_matrices(data1, data2,length,width):
        matrix = np.zeros((length, width,1), dtype=np.uint8)
        matrix= np.where((data1==1 )| (data2==1),1,matrix)
        return matrix

          
#this function plots runs with and their meassurment area
def plot_met_gebieden(all_runs, height, width, gebieden, min_size=1):
    
    # bouw afbeelding op
    image = np.ones((height, width, 3))
    
    valid_labels = set(
        node.label for node in all_runs
        if node.head is node and node.tot_area >= min_size
    )
    
    random.seed(42)
    color_map = {
        label: (random.random(), random.random(), random.random())
        for label in valid_labels
    }
    
    for node in all_runs:
        if node.label in valid_labels:
            color = color_map[node.label]
            image[node.row, node.start:node.end + 1] = color
    
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(image)
    
    # teken elk gebied als gekleurde rechthoek
    gebied_kleuren = ['red', 'blue', 'green', 'orange', 'purple']
    
    for i, gebied in enumerate(gebieden):
        kleur = gebied_kleuren[i % len(gebied_kleuren)]
        
        rect = mpatches.Rectangle(
            (gebied['col_min'], gebied['row_min']),          # x, y
            gebied['col_max'] - gebied['col_min'],           # breedte
            gebied['row_max'] - gebied['row_min'],           # hoogte
            linewidth  = 2,
            edgecolor  = kleur,
            facecolor  = kleur,
            alpha      = 0.2                                 # transparantie
        )
        ax.add_patch(rect)
        
        # naam van het gebied erbij
        ax.text(
            gebied['col_min'] + 5,
            gebied['row_min'] + 15,
            gebied['naam'],
            color     = kleur,
            fontsize  = 10,
            fontweight= 'bold'
        )
    
    # legenda voor de gebieden
    legenda = [
        mpatches.Patch(
            facecolor = gebied_kleuren[i % len(gebied_kleuren)],
            alpha     = 0.5,
            label     = gebied['naam']
        )
        for i, gebied in enumerate(gebieden)
    ]
    ax.legend(handles=legenda, bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.set_title("Connected Components met gebieden")
    ax.set_xticks(np.arange(0, width, 50))
    ax.set_yticks(np.arange(0, height, 50))
    
    plt.tight_layout()
    plt.show()

from skimage.transform import hough_circle, hough_circle_peaks
from skimage.feature import canny
import numpy as np

#this function detects the circels
def detecteer_rondjes(matrix, straal, tolerantie=10, drempel=0.5):
    """
    matrix    : numpy array van shape (h, w, 1) of (h, w)
    straal    : gezochte straal in pixels, bijv. 100 of 400
    tolerantie: zoekbereik rondom de straal (straal ± tolerantie)
    """
    # squeeze naar 2D als de matrix (x, x, 1) is
    if matrix.ndim == 3:
        matrix = matrix[:, :, 0]

    # zorg dat de waarden tussen 0 en 1 liggen
    matrix = (matrix - matrix.min()) / (matrix.max() - matrix.min() + 1e-8)

    # detecteer randen
    randen = canny(matrix, sigma=2.0, low_threshold=0.1, high_threshold=0.3)

    # zoek alleen in het bereik straal ± tolerantie
    radii = np.arange(max(1, straal - tolerantie), straal + tolerantie + 1, 1)
    hough_res = hough_circle(randen, radii)

    # pak de beste rondjes
    accums, cx, cy, radii_gevonden = hough_circle_peaks(
        hough_res,
        radii,
        min_xdistance=straal,
        min_ydistance=straal,
        threshold=drempel * hough_res.max()
    )

    rondjes = [
        {'x': int(x), 'y': int(y), 'r': int(r), 'score': float(a)}
        for a, x, y, r in zip(accums, cx, cy, radii_gevonden)
    ]

    return rondjes

In [ ]:
#this finds all clusters at once
#Creates a dataframe to save the output
df= pd.DataFrame({'well':[], 'dag':[], 'elektrode':[], 'cluster':[], 'pixels in gebied':[], 'afstand cluster tot elektrode':[]})
df2= pd.DataFrame({'well':[], 'dag':[], 'cluster':[], 'pixels in gebied':[]})
#Select all the wells I want to look at
wells=['A1','A2','A3','B1','B2','B3','C1','C2','C3','C4','C5','C6','D1','D2','D3','D4','D5','D6']
#List all the measuring days
dagen=[1,2,4,8]

for well in wells:
   for day in dagen:
    path= r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni_fotos\h5_bestanden"
    path= os.path.join(path, f"{well}\cropped_{well}_{day}_elektrode_cellen_Simple_Segmentation.h5")
    output_path= r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni_fotos\resultaten"
    output_path= os.path.join(output_path,f"{well}")

    with h5py.File(f"{path}") as f:
            data = f["exported_data"][:]    
            data= np.where(data==2,0,data)


    length, width, total, cel_fraction= fraction(data)
    # print(cel_fraction)

    # segmentatie_plot(data)

    image = np.squeeze(data)
    matrix2= pad_image(image)

    RowSegments= finding_all_runs(matrix2)

    connected_components_labeling(RowSegments, length,1, 4)

    test= assign_labels(RowSegments)

    #plot_connected_components(test,length, width)

    #Creats a new binary matrix where all components that are smaler than the threshold are filterd out
    checken2= create_binary_matrix(RowSegments, length, width, round((cel_fraction/0.0014),0))
    # print(checken2.shape)
    # segmentatie_plot(checken2)

    image2 = np.squeeze(checken2)
    matrix3= pad_image(image2)

    RowSegments2= finding_all_runs(matrix3)

    connected_components_labeling(RowSegments2, length,1, 5)

    test2= assign_labels(RowSegments2)

    #plot_connected_components(test2,length, width)

    
    path2= r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni_fotos\h5_bestanden"
    path2= os.path.join(path2, f"{well}\cropped_{well}_{day}_elektrode_Simple_Segmentation.h5")


    with h5py.File(f"{path2}") as f:
            data2 = f["exported_data"][:]   
            data2= np.where(data2==2,0,data2)
    image2 = np.squeeze(data2)
    matrix4= pad_image(image2)


    RowSegments4= finding_all_runs(matrix4)

    connected_components_labeling(RowSegments4, length,1, 3)

    test4= assign_labels(RowSegments4)

    # plot_connected_components(test4,length, width)

    #Filterd out small parts of the electrodes like lose pixels 
    electrodes= create_binary_matrix(RowSegments4, length, width, 2000) 
    # # print(electrodes.shape)
    # # segmentatie_plot(electrodes)

    image5 = np.squeeze(electrodes)
    matrix5= pad_image(image5)


    RowSegments5= finding_all_runs(matrix5)

    connected_components_labeling(RowSegments5, length,1, 3)

    test5= assign_labels(RowSegments5)

  
    checken3= create_binary_matrix(RowSegments2, length, width, 600)
   
    new_matrix= merging_matrices(data2,checken3,length,width)
    segmentatie_plot(new_matrix)

    clusters_nodes=[]
    for node in RowSegments2.copy():
        
        if node is node.head and node.tot_area>=700:
            clusters_nodes.append(node)
            
    clusters_runs = []

    for node in clusters_nodes:  # all_nodes bevat je head nodes
            walk = node
            while walk is not None:
                clusters_runs.append(walk)
                walk = walk.next


    plot_connected_components(clusters_runs,length, width)

    # gebruik — twee losse aanroepen voor verschillende stralen
    kleine_rondjes = detecteer_rondjes(data2, straal=50, tolerantie=15)
    grote_rondjes  = detecteer_rondjes(data2, straal=115, tolerantie=30)
    # of combineer ze
    alle_rondjes = kleine_rondjes + grote_rondjes
    # print(alle_rondjes)
    grotere_rondjes= [item for item in alle_rondjes if item["score"] > 0.23]
    
    max_afstand=20
    # Y-groepering
    y_gesorteerd = sorted(grotere_rondjes, key=lambda d: d['y'])
    y_groepen = []
    groep = [y_gesorteerd[0]]

    for a, b in zip(y_gesorteerd, y_gesorteerd[1:]):
        if b["y"] - a["y"] <= max_afstand:
            groep.append(b)
        else:
            y_groepen.append(groep)
            groep = [b]
    y_groepen.append(groep)

    y1 = round(sum(d['y'] for d in y_groepen[0]) / len(y_groepen[0]), 0)
    y2 = round(sum(d['y'] for d in y_groepen[1]) / len(y_groepen[1]), 0)
    y3 = round(sum(d['y'] for d in y_groepen[2]) / len(y_groepen[2]), 0)
    y4 = round(sum(d['y'] for d in y_groepen[3]) / len(y_groepen[3]), 0)

    # X-groepering
    x_gesorteerd = sorted(grotere_rondjes, key=lambda d: d['x'])
    x_groepen = []
    groep = [x_gesorteerd[0]]

    for a, b in zip(x_gesorteerd, x_gesorteerd[1:]):
        if b["x"] - a["x"] <= max_afstand:
            groep.append(b)
        else:
            x_groepen.append(groep)
            groep = [b]
    x_groepen.append(groep)

    x1 = round(sum(d['x'] for d in x_groepen[0]) / len(x_groepen[0]), 0)
    x2 = round(sum(d['x'] for d in x_groepen[1]) / len(x_groepen[1]), 0)
    x3 = round(sum(d['x'] for d in x_groepen[2]) / len(x_groepen[2]), 0)
    x4 = round(sum(d['x'] for d in x_groepen[3]) / len(x_groepen[3]), 0)

    gebieden3 = [
    {'naam': 'electrode 13', 'center_col': x1, 'center_row': y1, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 14', 'center_col': x2, 'center_row': y1, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 15', 'center_col': x3, 'center_row': y1, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 16', 'center_col': x4, 'center_row': y1, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 9',  'center_col': x1, 'center_row': y2, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 10', 'center_col': x2, 'center_row': y2, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 11', 'center_col': x3, 'center_row': y2, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 12', 'center_col': x4, 'center_row': y2, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 5',  'center_col': x1, 'center_row': y3, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 6',  'center_col': x2, 'center_row': y3, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 7',  'center_col': x3, 'center_row': y3, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 8',  'center_col': x4, 'center_row': y3, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 1',  'center_col': x1, 'center_row': y4, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 2',  'center_col': x2, 'center_row': y4, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 3',  'center_col': x3, 'center_row': y4, 'radius': 44, 'min_pixels': 10000},
    {'naam': 'electrode 4',  'center_col': x_gesorteerd[-1]['x'], 'center_row': y_gesorteerd[-1]['y'], 'radius': 142, 'min_pixels': 20000},
    ]

    gebieden4 = [
    {'naam': 'electrode 13', 'center_col': x1, 'center_row': y1, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 14', 'center_col': x2, 'center_row': y1, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 15', 'center_col': x3, 'center_row': y1, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 16', 'center_col': x4, 'center_row': y1, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 9',  'center_col': x1, 'center_row': y2, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 10', 'center_col': x2, 'center_row': y2, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 11', 'center_col': x3, 'center_row': y2, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 12', 'center_col': x4, 'center_row': y2, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 5',  'center_col': x1, 'center_row': y3, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 6',  'center_col': x2, 'center_row': y3, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 7',  'center_col': x3, 'center_row': y3, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 8',  'center_col': x4, 'center_row': y3, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 1',  'center_col': x1, 'center_row': y4, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 2',  'center_col': x2, 'center_row': y4, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 3',  'center_col': x3, 'center_row': y4, 'radius': 176, 'min_pixels': 10000},
    {'naam': 'electrode 4',  'center_col': x_gesorteerd[-1]['x'], 'center_row': y_gesorteerd[-1]['y'], 'radius': 176, 'min_pixels': 20000},
    ]       

    cirkel='meetoppervlak'
    plot_met_gebieden(RowSegments5,length,width, gebieden4)
    cirkel='elektrode_oppervlak'
    plot_met_gebieden(RowSegments5,length,width, gebieden3)


    for node in clusters_nodes:
        for i, gebied in enumerate(gebieden4):
            gebied3 = gebieden3[i]
            
            # snelle pre-filter: bounding box van de grote cirkel
            if not (node.col_max < gebied['center_col'] - gebied['radius'] or 
                    node.col_min > gebied['center_col'] + gebied['radius']):
                if not (node.row_max < gebied['center_row'] - gebied['radius'] or 
                        node.row_min > gebied['center_row'] + gebied['radius']):
                    
                    # tel pixels binnen de grote cirkel (gebieden)
                    pixels_in_gebied = 0
                    walk = node
                    while walk is not None:
                        for col in range(walk.start, walk.end + 1):
                            dist_sq = (col - gebied['center_col'])**2 + (walk.row - gebied['center_row'])**2
                            if dist_sq <= gebied['radius']**2:
                                pixels_in_gebied += 1
                        walk = walk.next
                    
                    # afstand middelpunt node tot rand van gebied2 (subcirkel)
                    node_center_col = (node.col_min + node.col_max) / 2
                    node_center_row = (node.row_min + node.row_max) / 2

                    afstand_tot_middelpunt = ((node_center_col - gebied3['center_col'])**2 + 
                                            (node_center_row - gebied3['center_row'])**2) ** 0.5

                    afstand_tot_rand = afstand_tot_middelpunt - gebied3['radius']

                    if pixels_in_gebied >= 40:
                        print(f"Rond om {gebied['naam']} ligt een cluster van {node.tot_area} pixels",
                             f"In het gebied liggen {pixels_in_gebied} pixels")
                        df.loc[len(df)] = [well, day, gebied['naam'], node.tot_area, pixels_in_gebied, afstand_tot_rand]


#df.to_csv('gevonden_clusters.csv', index=False, sep=",")
df.to_csv(r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni_fotos\resultaten\gevonden_clusters4.csv", index=False, sep=",")


        





